In [1]:
# Pyomo is an algebraic modeling language for Python. The `gdp` submodule
# adds Disjunction blocks for native generalized-disjunctive-programming
# modeling; a TransformationFactory step later rewrites them into a
# standard MILP that any LP/MIP solver can take.
import pyomo.environ as pyo
from pyomo.gdp import Disjunction

Sets

$\mathcal{I} = \{1, \ldots, N\}$ rectangles

Parameters

$w_i$ width of rectangle $i \in \mathcal{I}$ \
$\ell_i$ length of rectangle $i \in \mathcal{I}$ \
$W$ strip width (fixed)

Variables

$x_i, y_i \ge 0$ near-corner coordinates of rectangle $i$ \
$L \ge 0$ strip length (objective)

Objective and Constraints

\begin{gather}
 \min_{x, y, L} \; L \\
 \textrm{s.t.} \quad x_i + w_i \le W \quad \forall i \in \mathcal{I} \\
 y_i + \ell_i \le L \quad \forall i \in \mathcal{I} \\
 x_i, y_i, L \ge 0
\end{gather}

Disjunction (no overlap)

For every pair $(i, j)$ with $i < j$, at least one of four geometric separations must hold &mdash; rectangle $i$ must lie to the left of, right of, below, or above rectangle $j$:

\begin{gather}
 [x_i + w_i \le x_j] \;\vee\; [x_j + w_j \le x_i] \;\vee\; [y_i + \ell_i \le y_j] \;\vee\; [y_j + \ell_j \le y_i]
\end{gather}

In [2]:
# Problem instance. Eight rectangles with mixed shapes -- some tall, some
# wide, some square -- so the optimal packing is visually interesting.
# Strip width W = 10.
data = {
    "rects":  [1, 2, 3, 4, 5, 6, 7, 8],
    "w":      {1: 2.0, 2: 3.0, 3: 4.0, 4: 5.0, 5: 2.0, 6: 3.0, 7: 6.0, 8: 1.0},
    "length": {1: 6.0, 2: 4.0, 3: 2.0, 4: 3.0, 5: 5.0, 6: 3.0, 7: 2.0, 8: 7.0},
    "W":      10.0,
}

In [3]:
# Build the GDP model.
# ConcreteModel binds parameter values at construction time (as opposed to
# AbstractModel, which is parameterized and only instantiated later).
m = pyo.ConcreteModel()

rects = list(data["rects"])
W = float(data["W"])

# Loose upper bound on the strip length: stacking every rectangle end-to-end
# along L is a (worst-case) feasible packing. Used as both a y-coordinate
# bound and an L bound so the gdp.bigm transformation can derive a sensible
# Big-M automatically from the variable bounds.
L_max = float(sum(data["length"][i] for i in rects))

# Index set over rectangles.
m.RECTS = pyo.Set(initialize=rects, ordered=True)

# Parameters: data the solver doesn't change.
m.w = pyo.Param(m.RECTS, initialize={i: data["w"][i] for i in rects})
m.length = pyo.Param(m.RECTS, initialize={i: data["length"][i] for i in rects})
m.W = pyo.Param(initialize=W)

# Decision variables. Bounds are explicit so the GDP transformation can
# pick a tight Big-M for each disjunct.
m.x = pyo.Var(m.RECTS, domain=pyo.NonNegativeReals, bounds=(0.0, W))
m.y = pyo.Var(m.RECTS, domain=pyo.NonNegativeReals, bounds=(0.0, L_max))
m.L = pyo.Var(domain=pyo.NonNegativeReals, bounds=(0.0, L_max))

# Objective: minimize strip length.
m.total_length = pyo.Objective(expr=m.L, sense=pyo.minimize)

# Containment: every rectangle fits inside the strip in both directions.
m.fit_x = pyo.Constraint(m.RECTS, rule=lambda m, i: m.x[i] + m.w[i] <= m.W)
m.fit_y = pyo.Constraint(m.RECTS, rule=lambda m, i: m.y[i] + m.length[i] <= m.L)

# Non-overlap disjunctions. For every unordered pair (i, j) with i < j,
# the rectangles must be separated in at least one of four ways. A
# `Disjunction` accepts a list of disjuncts, each being a list of
# constraint expressions; the GDP transformation later picks one and
# enforces it via Big-M / Hull / etc.
pairs = [(i, j) for idx, i in enumerate(rects) for j in rects[idx + 1:]]
m.PAIRS = pyo.Set(initialize=pairs, dimen=2)

def disj_rule(m, i, j):
    return [
        [m.x[i] + m.w[i] <= m.x[j]],            # i left of j
        [m.x[j] + m.w[j] <= m.x[i]],            # i right of j
        [m.y[i] + m.length[i] <= m.y[j]],       # i below j
        [m.y[j] + m.length[j] <= m.y[i]],       # i above j
    ]
m.no_overlap = Disjunction(m.PAIRS, rule=disj_rule)

In [4]:
# Reformulate the GDP into a standard MILP. Big-M is the classical choice --
# fewest variables, often the loosest LP relaxation. Other transformations
# available in pyomo.gdp:
#   gdp.hull         -- disaggregated variables, tighter LP relaxation
#   gdp.mbigm        -- per-constraint tight Big-M (midpoint of bigm and hull)
#   gdp.cuttingplane -- iteratively adds violated hull facets to a Big-M base
pyo.TransformationFactory("gdp.bigm").apply_to(m)

# Solve with HiGHS (ships as a pip wheel via highspy).
# tee=True streams the solver log to stdout so we can see HiGHS's branch-
# and-bound progress.
solver = pyo.SolverFactory("appsi_highs")
results = solver.solve(m, tee=True)

# Optimal strip length and per-rectangle near-corner coordinates.
print(f"\nOptimal strip length L = {pyo.value(m.L):.3f}")
print(f"\n{'rect':>4}  {'x':>6}  {'y':>6}  {'w':>4}  {'length':>6}")
for i in rects:
    print(
        f"{i:>4}  "
        f"{pyo.value(m.x[i]):>6.2f}  "
        f"{pyo.value(m.y[i]):>6.2f}  "
        f"{data['w'][i]:>4.1f}  "
        f"{data['length'][i]:>6.1f}"
    )

Running HiGHS 1.14.0 (git hash: 7df0786): Copyright (c) 2026 under MIT licence terms
MIP has 156 rows; 129 cols; 472 nonzeros; 112 integer variables (112 binary)
Coefficient ranges:
  Matrix  [1e+00, 4e+01]
  Cost    [1e+00, 1e+00]
  Bound   [1e+00, 3e+01]
  RHS     [1e+00, 3e+01]
Presolving model
146 rows, 127 cols, 456 nonzeros 0s
144 rows, 126 cols, 451 nonzeros 0s
Presolve reductions: rows 144(-12); columns 126(-3); nonzeros 451(-21) 

Solving MIP model with:
   144 rows
   126 cols (109 binary, 0 integer, 0 implied int., 17 continuous, 0 domain fixed)
   451 nonzeros

Src: B => Branching; C => Central rounding; F => Feasibility pump; H => Heuristic;
     I => Shifting; J => Feasibility jump; L => Sub-MIP; P => Empty MIP; R => Randomized rounding;
     S => Solve LP; T => Evaluate node; U => Unbounded; X => User solution; Y => HiGHS solution;
     Z => ZI Round; l => Trivial lower; p => Trivial point; u => Trivial upper; z => Trivial zero

        Nodes      |    B&B Tree     |    

 L      69       7        16   7.03%   7               10                30.00%     1827     79     46      1151     0.2s
 T     373      27       117  21.45%   7               10                30.00%     2583     52    788      5886     0.3s


 L    2148      54       726  31.42%   7               9                 22.22%     4313     56   3933     30484     2.9s


      2955       0      1056 100.00%   8.999999        9                  0.00%     4970    106   5919     47388     4.7s

Solving report
  Status            Optimal
  Primal bound      9
  Dual bound        8.999999
  Gap               1.1e-05% (tolerance: 0.01%)
  P-D integral      1.31177135463
  Solution status   feasible
                    9 (objective)
                    0 (bound viol.)
                    0 (int. viol.)
                    0 (row viol.)
  Timing            4.75
  Max sub-MIP depth 3
  Nodes             2955
  Repair LPs        0
  LP iterations     47388
                    6963 (strong br.)
                    5317 (separation)
                    2805 (heuristics)

Optimal strip length L = 9.000

rect       x       y     w  length
   1    0.00    3.00   2.0     6.0
   2    3.00    0.00   3.0     4.0
   3    6.00    0.00   4.0     2.0
   4    2.00    4.00   5.0     3.0
   5    7.00    2.00   2.0     5.0
   6    0.00    0.00   3.0     3.0
   7    2.00    7.00 